[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/21_dimensionality_reduction/02_pca_in_practice.ipynb)

# Bài 02 — PCA trong thực tế

**🎯 Mục tiêu**
- **Scaling** trước PCA — vì sao PCA trên dữ liệu chưa chuẩn hoá chỉ "thấy" feature có thang đo lớn.
- PCA để **trực quan hoá**, **nén** (tái tạo ảnh chữ số), **khử nhiễu**, và làm **tiền xử lý** (chọn số thành phần bằng CV).
- Khi PCA **làm hại**: tín hiệu dự đoán nằm ở hướng phương sai **nhỏ**.
- Biến thể: Kernel PCA (phi tuyến), TruncatedSVD (dữ liệu thưa), IncrementalPCA (dữ liệu lớn).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/21_dimensionality_reduction

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits, make_circles
from sklearn.decomposition import PCA, IncrementalPCA, KernelPCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course.datasets import make_house_prices
from ml_course.visualization import PALETTE, plot_images, set_style

set_style()
rng = np.random.default_rng(0)
digits = load_digits()
X, y = digits.data, digits.target

## 1. Scaling trước PCA

PCA tìm hướng phương sai lớn nhất → feature có **đơn vị lớn** (phương sai lớn) chiếm trọn PC1, bất kể có ý nghĩa hay không.

In [ ]:
houses = make_house_prices(n=1000, seed=0)
num = ["area_m2", "bedrooms", "bathrooms", "age_years", "distance_to_center_km"]
H = houses[num].assign(area_m2=lambda d: d.area_m2 * 10_000)              # diện tích đổi sang cm²!
for name, Xh in [("không scale", H.to_numpy()), ("StandardScaler", StandardScaler().fit_transform(H))]:
    p = PCA(2).fit(Xh)
    load = pd.Series(np.abs(p.components_[0]), index=num).round(3)
    print(f"{name:15s} PC1 giải thích {p.explained_variance_ratio_[0]:.1%} phương sai | |trọng số PC1|: {load.to_dict()}")

Không scale: PC1 ≈ trục diện tích (99.99% "phương sai" chỉ vì đơn vị cm²). Sau khi chuẩn hoá: PC1 là tổ hợp của diện tích, số phòng ngủ, phòng tắm — một trục "**kích thước nhà**" có ý nghĩa. (Ngoại lệ: các feature cùng đơn vị và thang đo có ý nghĩa — vd pixel ảnh — thường **không** scale.)

## 2. Trực quan hoá: 64 chiều → 2 chiều

In [ ]:
Z = PCA(2).fit_transform(X)
fig, ax = plt.subplots(figsize=(8, 6.5))
for d in range(10):
    ax.scatter(*Z[y == d].T, s=8, alpha=0.6, color=plt.cm.tab10(d), label=str(d))
ax.legend(markerscale=2, ncol=2, fontsize=8)
ax.set_title("Digits chiếu lên 2 thành phần chính (PCA tuyến tính) — các lớp chồng lấn nhiều")
plt.show()

## 3. Nén và tái tạo ảnh

Mỗi ảnh 8×8 = 64 số. Giữ $k$ thành phần → mỗi ảnh chỉ còn $k$ số; tái tạo bằng `inverse_transform`.

In [ ]:
fig, axes = plt.subplots(5, 10, figsize=(12, 6.5))
for row, k in enumerate([2, 5, 10, 20, 64]):
    p = PCA(k).fit(X)
    rec = p.inverse_transform(p.transform(X[:10]))
    for col in range(10):
        axes[row, col].imshow(rec[col].reshape(8, 8), cmap="gray_r")
        axes[row, col].axis("off")
    axes[row, 0].set_title(f"k = {k} ({p.explained_variance_ratio_.sum():.0%})", fontsize=8, loc="left")
plt.suptitle("Tái tạo ảnh chữ số từ k thành phần chính", y=1.0)
plt.tight_layout()
plt.show()

Các thành phần chính của ảnh ("eigendigits") là những "mẫu" cơ bản; mỗi ảnh ≈ ảnh trung bình + tổ hợp tuyến tính của chúng:

In [ ]:
p10 = PCA(10).fit(X)
plot_images(np.vstack([p10.mean_, p10.components_]).reshape(-1, 8, 8), labels=["trung bình"] + [f"PC{i + 1}" for i in range(10)], n=11, ncols=11)
plt.show()

## 4. Khử nhiễu

Nhiễu ngẫu nhiên phân tán đều trên mọi hướng, còn tín hiệu tập trung ở vài hướng đầu → chiếu lên $k$ thành phần đầu rồi tái tạo sẽ **loại bỏ phần lớn nhiễu**.

In [ ]:
X_noisy = X + rng.normal(0, 4, X.shape)
p_dn = PCA(0.5).fit(X_noisy)                       # giữ 50% phương sai của dữ liệu nhiễu
X_dn = p_dn.inverse_transform(p_dn.transform(X_noisy))
print(f"giữ {p_dn.n_components_} thành phần | MSE so với ảnh sạch: nhiễu = {np.mean((X_noisy - X) ** 2):.2f}, sau khử nhiễu = {np.mean((X_dn - X) ** 2):.2f}")
fig, axes = plt.subplots(3, 8, figsize=(10, 4))
for col in range(8):
    for row, img in enumerate([X, X_noisy, X_dn]):
        axes[row, col].imshow(img[col].reshape(8, 8), cmap="gray_r")
        axes[row, col].axis("off")
for row, name in enumerate(["sạch", "nhiễu", "khử nhiễu"]):
    axes[row, 0].set_title(name, fontsize=8, loc="left")
plt.show()

## 5. PCA làm tiền xử lý — chọn số thành phần bằng CV

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=0)
for name, model in [("KNN", KNeighborsClassifier(5)), ("LogReg", LogisticRegression(max_iter=3000))]:
    res = []
    for k in [5, 10, 20, 40, None]:
        steps = [StandardScaler()] if name == "LogReg" else []
        pipe = make_pipeline(*steps, PCA(k) if k else "passthrough", model)
        acc = cross_val_score(pipe, X, y, cv=cv).mean()
        res.append(f"k={k or 64}: {acc:.4f}")
    print(f"{name:7s} | " + " | ".join(res))

Với **KNN**, 20–40 thành phần (thay vì 64) cho accuracy gần như bằng — 40 thành phần còn nhỉnh hơn dùng cả 64 (bớt chiều nhiễu, chương 08) và dự đoán nhanh hơn. Với **Logistic Regression**, giảm mạnh số chiều làm mất thông tin hữu ích (20 thành phần: ~0.94 so với ~0.97). Số thành phần là **hyperparameter phụ thuộc model** — chọn bằng CV, PCA nằm **trong** Pipeline (fit trên train của từng fold).

## 6. Khi PCA làm hại

PCA **không nhìn nhãn** — nó giữ hướng phương sai lớn, không phải hướng **dự đoán tốt**. Nếu tín hiệu nằm ở hướng phương sai nhỏ, PCA vứt bỏ đúng thông tin cần thiết:

In [ ]:
n = 2000
big = rng.normal(0, 10, (n, 2))                   # 2 feature phương sai lớn, vô ích
small = rng.normal(0, 1, (n, 1))                  # 1 feature phương sai nhỏ, quyết định nhãn
Xh = np.hstack([big, small])
yh = (small.ravel() > 0).astype(int)
for k in [1, 2, 3]:
    acc = cross_val_score(make_pipeline(PCA(k), LogisticRegression()), Xh, yh, cv=5).mean()
    print(f"PCA giữ {k} thành phần: accuracy = {acc:.3f}")

→ Khi mục tiêu là **dự đoán**, cân nhắc phương pháp có giám sát (feature selection, PLS, LDA) hoặc để model regularized tự xử lý; luôn kiểm chứng PCA bằng CV.

## 7. Biến thể

In [ ]:
Xc, yc = make_circles(500, factor=0.4, noise=0.05, random_state=0)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].scatter(*Xc.T, c=[PALETTE[i] for i in yc], s=8)
axes[0].set_title("gốc")
axes[1].scatter(PCA(1).fit_transform(Xc).ravel(), np.zeros(500) + rng.normal(0, 0.02, 500), c=[PALETTE[i] for i in yc], s=8)
axes[1].set_title("PCA 1 chiều: hai lớp chồng lên nhau")
kp = KernelPCA(1, kernel="rbf", gamma=5).fit_transform(Xc)
axes[2].scatter(kp.ravel(), np.zeros(500) + rng.normal(0, 0.02, 500), c=[PALETTE[i] for i in yc], s=8)
axes[2].set_title("Kernel PCA (RBF) 1 chiều: tách được")
plt.show()

| Biến thể | Khi nào |
|---|---|
| **Kernel PCA** | cấu trúc phi tuyến (kernel trick — chương 13) |
| **TruncatedSVD** | ma trận thưa (TF-IDF văn bản — "LSA"); không trừ trung bình |
| **IncrementalPCA** | dữ liệu không vừa bộ nhớ — fit theo lô |
| `PCA(svd_solver="randomized")` | dữ liệu lớn, chỉ cần vài thành phần đầu |
| Sparse PCA, NMF, ICA | thành phần thưa/không âm/độc lập — dễ diễn giải hơn |

In [ ]:
ipca = IncrementalPCA(20, batch_size=200)
for batch in np.array_split(X, 9):
    ipca.partial_fit(batch)
print(f"IncrementalPCA (9 lô) giải thích {ipca.explained_variance_ratio_.sum():.4f} | PCA đầy đủ: {PCA(20).fit(X).explained_variance_ratio_.sum():.4f}")

## 🧠 Kiểm tra nhanh

1. Vì sao phải chuẩn hoá trước PCA khi feature có đơn vị khác nhau? Khi nào không cần?
2. PCA có dùng nhãn không? Hệ quả gì khi dùng làm tiền xử lý cho bài toán dự đoán?
3. Vì sao chiếu lên vài thành phần đầu rồi tái tạo lại có thể khử nhiễu?
4. Dữ liệu TF-IDF thưa 100 000 chiều: dùng PCA hay TruncatedSVD? Vì sao?

## 📌 Tóm tắt

- Scale trước PCA (trừ khi feature cùng thang đo); chọn số thành phần theo phương sai tích luỹ hoặc CV.
- Ứng dụng: trực quan hoá, nén, khử nhiễu, tiền xử lý (trong Pipeline).
- PCA không nhìn nhãn → có thể vứt tín hiệu; Kernel PCA/TruncatedSVD/IncrementalPCA cho các tình huống đặc biệt.

## 📚 Tài liệu tham khảo

- Jolliffe & Cadima (2016) — *Principal component analysis: a review and recent developments*
- Halko, Martinsson & Tropp (2011) — *Finding Structure with Randomness* (randomized SVD)
- scikit-learn User Guide — Decomposing signals in components